# MARS Walkthrough

This notebook demonstrates the full MARS pipeline interactively:

1. **Run the pipeline** — execute System 1 → System 2 ↔ System 3 for a single query
2. **Visualize the pipeline** — Gantt chart, performance metrics, flow diagram
3. **Inspect agent interactions** — chat logs, KG subgraphs, RAG results, iteration details

For batch execution of all queries, use the CLI scripts instead:
```bash
python scripts/run_mars.py          # full MARS pipeline
python scripts/run_ablations.py     # ablation conditions
python scripts/run_evaluation.py    # LLM-judge evaluation
```


## Part 1: Run the MARS Pipeline


### Setup


In [ ]:
import sys
import os
import json
from pathlib import Path

# Locate project root
current_dir = Path().resolve()
if (current_dir / "src").exists():
    project_root = current_dir
elif (current_dir.parent / "src").exists():
    project_root = current_dir.parent
else:
    project_root = Path(os.environ.get("SYS3DEV_ROOT", current_dir.parent))
sys.path.insert(0, str(project_root))
print(f"project_root: {project_root}")

from src.config import load_config
from src.utils.ablation_utils import load_ablation_queries
from src.runner import initialize, run_query

config = load_config()
queries = load_ablation_queries()
print(f"Loaded {len(queries)} queries: {[q['name'] for q in queries]}")


### Select a query


In [ ]:
# Pick which query to run (change the index to try a different one)
query = queries[0]  # Query1

print(f"Query:       {query['name']}")
print(f"Sentence:    {query['sentence'][:120]}…")
print(f"Material X:  {query['material_X']}")
print(f"Application: {query['application_Y']}")


### Initialize pipeline components

This loads knowledge graphs, ChromaDB collections, embedding models, and creates all agents.
It takes a few minutes on first run.


In [ ]:
components = initialize(config)


### Run the pipeline


In [ ]:
output_dir = str(project_root / "results" / query["name"])
pipeline_run = run_query(components, query, output_dir)


---

## Part 2: Pipeline Visualization

Visualize the pipeline run: execution timeline, performance metrics, and flow diagram.


### Load pipeline run data


In [ ]:
import json
import os
import glob
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
from IPython.display import display, HTML, Markdown
import networkx as nx
from collections import defaultdict


In [ ]:
def find_most_recent_pipeline_run(query_name=None):
    """Find the most recent pipeline_run JSON file, optionally for a specific query."""
    cwd = Path.cwd().resolve()
    candidate_dirs = []

    if query_name:
        for parent in [cwd, cwd.parent]:
            candidate_dirs.append(parent / "results" / query_name / "artifacts")
        for parent in cwd.parents:
            if (parent / "src").exists():
                candidate_dirs.append(parent / "results" / query_name / "artifacts")

    for parent in [cwd, cwd.parent]:
        candidate_dirs.append(parent / "pipeline_logs")
    for parent in cwd.parents:
        if (parent / "src").exists():
            candidate_dirs.append(parent / "pipeline_logs")

    env_dir = os.environ.get("PIPELINE_LOGS_DIR")
    if env_dir:
        candidate_dirs.append(Path(env_dir).expanduser().resolve())

    seen, unique = set(), []
    for p in candidate_dirs:
        if str(p) not in seen:
            unique.append(p)
            seen.add(str(p))

    runs = []
    for d in unique:
        runs.extend(glob.glob(str(d / "pipeline_run_*.json")))

    if not runs:
        raise FileNotFoundError("No pipeline_run files found.")

    runs.sort(key=os.path.getmtime, reverse=True)
    path = runs[0]
    run_id = os.path.basename(path).replace("pipeline_run_", "").replace(".json", "")
    return path, run_id


def load_pipeline_run(pipeline_run_path):
    """Load pipeline run and organize data by iteration."""
    with open(pipeline_run_path) as f:
        pr = json.load(f)

    logs_dir = os.path.dirname(pipeline_run_path)
    chats_dir = os.path.join(logs_dir, "chats")

    def _load(path):
        if path and os.path.exists(path):
            with open(path) as f:
                return json.load(f)
        return None

    s1 = pr.get("system1", {})
    s1_chat = _load(s1.get("chat_log_path") or os.path.join(chats_dir, f"system1_chat_log_material_requirements_{s1.get('run_id')}.json"))
    s1_result = _load(s1.get("result_path") or os.path.join(logs_dir, f"system1_{s1.get('run_id')}.json"))

    iterations = []
    for it in pr.get("system2_system3_loop", {}).get("iterations", []):
        s2, s3 = it.get("system2", {}), it.get("system3", {})
        iterations.append({
            "iteration": it.get("iteration"),
            "system2": {
                "run_id": s2.get("run_id"),
                "chat_log": _load(s2.get("chat_log_path") or os.path.join(chats_dir, f"system2_chat_log_material_discovery_{s2.get('run_id')}.json")),
                "result": _load(s2.get("result_path") or os.path.join(logs_dir, f"system2_{s2.get('run_id')}.json")),
                "candidate": s2.get("candidate", ""),
                "start_time": s2.get("start_time"),
                "end_time": s2.get("end_time"),
                "duration_seconds": s2.get("duration_seconds", 0),
                "internal_iterations": s2.get("internal_iterations", []),
            },
            "system3": {
                "run_id": s3.get("run_id"),
                "chat_log": _load(s3.get("chat_log_path") or os.path.join(chats_dir, f"system3_chat_log_manufacturability_assessment_{s3.get('run_id')}.json")),
                "result": _load(s3.get("result_path") or os.path.join(logs_dir, f"system3_{s3.get('run_id')}.json")),
                "status": s3.get("status", "unknown"),
                "start_time": s3.get("start_time"),
                "end_time": s3.get("end_time"),
                "duration_seconds": s3.get("duration_seconds", 0),
                "blocking_constraints": s3.get("blocking_constraints", []),
                "feedback_to_system2": s3.get("feedback_to_system2", ""),
            },
        })

    return {
        "pipeline_run_id": pr.get("pipeline_run_id"),
        "start_time": pr.get("start_time"),
        "end_time": pr.get("end_time"),
        "total_duration_seconds": pr.get("total_duration_seconds", 0),
        "system1": {
            "run_id": s1.get("run_id"),
            "chat_log": s1_chat,
            "result": s1_result,
            "start_time": s1.get("start_time"),
            "end_time": s1.get("end_time"),
            "duration_seconds": s1.get("duration_seconds", 0),
        },
        "iterations": iterations,
        "final_outcome": pr.get("final_outcome", {}),
    }


In [ ]:
# Load the pipeline run from Part 1 (or from frozen results)
try:
    pr_path, pr_id = find_most_recent_pipeline_run(query["name"])
except FileNotFoundError:
    pr_path, pr_id = find_most_recent_pipeline_run()

print(f"Pipeline run: {pr_id}")
print(f"Path: {pr_path}")

with open(pr_path) as f:
    pipeline_run_raw = json.load(f)
pipeline_data = load_pipeline_run(pr_path)

print(f"Duration: {pipeline_data['total_duration_seconds']:.0f}s")
print(f"Iterations: {len(pipeline_data['iterations'])}")
print(f"Status: {pipeline_data['final_outcome'].get('status', 'N/A')}")


### Timeline (Gantt Chart)


In [ ]:
def create_timeline_gantt(pipeline_run):
    """
    Create a Gantt chart showing the execution timeline.
    """
    fig, ax = plt.subplots(figsize=(16, 10))
    
    # Parse start time
    start_time = datetime.fromisoformat(pipeline_run['start_time'].replace('Z', '+00:00'))
    
    # Colors for different systems
    colors = {
        'system1': '#4A90E2',
        'system2': '#50C878',
        'system3': '#FF6B6B',
        'system2_internal': '#9B59B6'
    }
    
    y_pos = 0
    y_labels = []
    
    # Plot System 1
    if pipeline_run['system1']['start_time']:
        s1_start = datetime.fromisoformat(pipeline_run['system1']['start_time'].replace('Z', '+00:00'))
        s1_end = datetime.fromisoformat(pipeline_run['system1']['end_time'].replace('Z', '+00:00'))
        s1_duration = (s1_end - s1_start).total_seconds() / 60  # Convert to minutes
        
        ax.barh(y_pos, s1_duration, left=(s1_start - start_time).total_seconds() / 60, 
                color=colors['system1'], alpha=0.8, label='System 1')
        y_labels.append('System 1')
        y_pos += 1
    
    # Plot System 2 → System 3 iterations
    iterations = pipeline_run.get('system2_system3_loop', {}).get('iterations', []) or []
    for iter_data in iterations:
        iter_num = iter_data['iteration']
        
        # System 2
        s2_start = datetime.fromisoformat(iter_data['system2']['start_time'].replace('Z', '+00:00'))
        s2_end = datetime.fromisoformat(iter_data['system2']['end_time'].replace('Z', '+00:00'))
        s2_duration = (s2_end - s2_start).total_seconds() / 60  # Convert to minutes
        
        ax.barh(y_pos, s2_duration, left=(s2_start - start_time).total_seconds() / 60,
                color=colors['system2'], alpha=0.8, label=f'System 2 (Iter {iter_num})' if iter_num == 1 else '')
        y_labels.append(f'System 2 (Iter {iter_num})')
        
        # Plot internal iterations within System 2
        internal_y = y_pos + 0.3
        for internal_iter in iter_data['system2'].get('internal_iterations', []):
            int_start = datetime.fromisoformat(internal_iter['start_time'].replace('Z', '+00:00'))
            int_end = datetime.fromisoformat(internal_iter['end_time'].replace('Z', '+00:00'))
            int_duration = (int_end - int_start).total_seconds() / 60  # Convert to minutes
            
            ax.barh(internal_y, int_duration, left=(int_start - start_time).total_seconds() / 60,
                    height=0.2, color=colors['system2_internal'], alpha=0.6)
            internal_y += 0.25
        
        y_pos += 1
        
        # System 3
        s3_start = datetime.fromisoformat(iter_data['system3']['start_time'].replace('Z', '+00:00'))
        s3_end = datetime.fromisoformat(iter_data['system3']['end_time'].replace('Z', '+00:00'))
        s3_duration = (s3_end - s3_start).total_seconds() / 60  # Convert to minutes
        
        status_color = colors['system3']
        if iter_data['system3']['status'] == 'manufacturable':
            status_color = '#2ECC71'  # Green for success
        
        ax.barh(y_pos, s3_duration, left=(s3_start - start_time).total_seconds() / 60,
                color=status_color, alpha=0.8, label=f'System 3 (Iter {iter_num})' if iter_num == 1 else '')
        y_labels.append(f"System 3 (Iter {iter_num}) - {iter_data['system3']['status']}")
        y_pos += 1
    
    # Formatting
    ax.set_yticks(range(len(y_labels)))
    ax.set_yticklabels(y_labels)
    ax.set_xlabel('Time (minutes from start)', fontsize=12)
    ax.set_title('Pipeline Execution Timeline', fontsize=16, fontweight='bold', pad=20)
    ax.grid(axis='x', alpha=0.3)
    
    # Invert y-axis so System 1 is at the top and System 3 at the bottom
    ax.invert_yaxis()
    
    # Create custom legend with all categories
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=colors['system1'], alpha=0.8, label='System 1'),
        Patch(facecolor=colors['system2'], alpha=0.8, label='System 2'),
        Patch(facecolor=colors['system2_internal'], alpha=0.6, label='System 2 Internal'),
        Patch(facecolor=colors['system3'], alpha=0.8, label='System 3 (Blocked)'),
        Patch(facecolor='#2ECC71', alpha=0.8, label='System 3 (Manufacturable)')
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    return fig

# Create timeline visualization
fig = create_timeline_gantt(pipeline_run)
plt.show()

create_timeline_gantt(pipeline_run_raw)


### Performance Metrics


In [ ]:
# Calculate and display performance metrics
total_duration = pipeline_run.get('total_duration_seconds', 0) or 0
s1_duration = pipeline_run.get('system1', {}).get('duration_seconds', 0) or 0

s2_durations = []
s3_durations = []
internal_iter_durations = []

for iter_data in (pipeline_run.get('system2_system3_loop', {}).get('iterations', []) or []):
    s2_durations.append(iter_data['system2']['duration_seconds'])
    s3_durations.append(iter_data['system3']['duration_seconds'])
    
    for internal_iter in iter_data['system2'].get('internal_iterations', []):
        internal_iter_durations.append(internal_iter['duration_seconds'])

metrics_html = f"""
<div style='font-family: Arial, sans-serif; padding: 20px; background: #f5f5f5; border-radius: 10px;'>
    <h2 style='color: #333; margin-top: 0;'>Performance Metrics</h2>
    
    <div style='display: grid; grid-template-columns: repeat(2, 1fr); gap: 20px;'>
        <div style='background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);'>
            <h3 style='color: #4A90E2; margin-top: 0;'>Total Execution</h3>
            <p style='font-size: 24px; font-weight: bold; color: #333;'>{total_duration:.2f} seconds</p>
            <p style='color: #666;'>({total_duration/60:.2f} minutes)</p>
        </div>
        
        <div style='background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);'>
            <h3 style='color: #4A90E2; margin-top: 0;'>System 1</h3>
            <p style='font-size: 24px; font-weight: bold; color: #333;'>{s1_duration:.2f} seconds</p>
            <p style='color: #666;'>({(s1_duration/total_duration*100) if total_duration else 0:.1f}% of total)</p>
        </div>
        
        <div style='background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);'>
            <h3 style='color: #50C878; margin-top: 0;'>System 2 (Average)</h3>
            <p style='font-size: 24px; font-weight: bold; color: #333;'>{np.mean(s2_durations) if s2_durations else 0:.2f} seconds</p>
            <p style='color: #666;'>Total: {sum(s2_durations):.2f}s over {len(s2_durations)} iteration(s)</p>
        </div>
        
        <div style='background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);'>
            <h3 style='color: #FF6B6B; margin-top: 0;'>System 3 (Average)</h3>
            <p style='font-size: 24px; font-weight: bold; color: #333;'>{np.mean(s3_durations) if s3_durations else 0:.2f} seconds</p>
            <p style='color: #666;'>Total: {sum(s3_durations):.2f}s over {len(s3_durations)} iteration(s)</p>
        </div>
    </div>
    
    {f"<div style='background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); margin-top: 20px;'>"
     f"<h3 style='color: #9B59B6; margin-top: 0;'>System 2 Internal Iterations (Average)</h3>"
     f"<p style='font-size: 24px; font-weight: bold; color: #333;'>{np.mean(internal_iter_durations) if internal_iter_durations else 0:.2f} seconds</p>"
     f"<p style='color: #666;'>Total: {len(internal_iter_durations)} internal iteration(s)</p>"
     f"</div>" if internal_iter_durations else ""}
</div>
"""

display(HTML(metrics_html))

### Pipeline Flow Diagram


In [ ]:
def create_flow_diagram(pipeline_run):
    """Create a flow diagram showing the pipeline execution flow."""

    def _safe_upper(val, default="N/A"):
        if val is None or val == "":
            return default
        return str(val).upper()

    def _safe_trunc(val, n, default="N/A"):
        if val is None or val == "":
            return default
        s = str(val)
        return (s[:n] + "...") if len(s) > n else s

    fig, ax = plt.subplots(figsize=(14, 10))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')

    # Colors
    s1_color = '#4A90E2'
    s2_color = '#50C878'
    success_color = '#2ECC71'
    blocked_color = '#E74C3C'
    neutral_color = '#95A5A6'

    y_start = 9
    x_center = 5

    # System 1
    s1_box = Rectangle((x_center - 1, y_start - 0.5), 2, 0.8,
                       facecolor=s1_color, alpha=0.8, edgecolor='black', linewidth=2)
    ax.add_patch(s1_box)
    ax.text(x_center, y_start, 'System 1\nProperty Extraction',
            ha='center', va='center', fontsize=10, fontweight='bold', color='white')

    # Arrow down
    ax.arrow(x_center, y_start - 0.5, 0, -0.5, head_width=0.2, head_length=0.1,
             fc='black', ec='black')

    y_pos = y_start - 1.5

    # System 2 → System 3 loop
    loop_info = pipeline_run.get('system2_system3_loop', {})
    iterations = loop_info.get('iterations', []) or []

    for i, iter_data in enumerate(iterations):
        iter_num = iter_data.get('iteration', i + 1)

        # System 2 box
        s2_box = Rectangle((x_center - 1, y_pos - 0.5), 2, 0.8,
                           facecolor=s2_color, alpha=0.8, edgecolor='black', linewidth=2)
        ax.add_patch(s2_box)
        candidate = _safe_trunc(iter_data.get('system2', {}).get('candidate'), 30)
        ax.text(x_center, y_pos, f'System 2 (Iter {iter_num})\n{candidate}',
                ha='center', va='center', fontsize=9, fontweight='bold', color='white')

        # Arrow to System 3
        ax.arrow(x_center, y_pos - 0.5, 0, -0.5, head_width=0.2, head_length=0.1,
                 fc='black', ec='black')

        y_pos -= 1.5

        # System 3 box
        s3_status = iter_data.get('system3', {}).get('status')
        s3_box_color = success_color if s3_status == 'manufacturable' else blocked_color
        s3_box = Rectangle((x_center - 1, y_pos - 0.5), 2, 0.8,
                           facecolor=s3_box_color, alpha=0.8, edgecolor='black', linewidth=2)
        ax.add_patch(s3_box)
        ax.text(x_center, y_pos, f'System 3 (Iter {iter_num})\n{_safe_upper(s3_status, "UNKNOWN")}',
                ha='center', va='center', fontsize=9, fontweight='bold', color='white')

        # Loop-back arrows
        if s3_status == 'blocked' and i < len(iterations) - 1:
            ax.arrow(x_center - 1, y_pos - 0.5, -1.5, 0, head_width=0.15, head_length=0.1,
                     fc='red', ec='red', linestyle='--', linewidth=2)
            ax.arrow(x_center - 2.5, y_pos - 0.5, 0, 3, head_width=0.15, head_length=0.1,
                     fc='red', ec='red', linestyle='--', linewidth=2)
            ax.text(x_center - 3, y_pos + 1, 'Loop Back', rotation=90,
                    ha='center', va='center', fontsize=8, color='red', fontweight='bold')

        y_pos -= 1.5

    if not iterations:
        # Display a placeholder when no iterations have been executed
        no_iter_box = Rectangle((x_center - 2, y_pos - 0.5), 4, 0.8,
                               facecolor=neutral_color, alpha=0.8, edgecolor='black', linewidth=2)
        ax.add_patch(no_iter_box)
        ax.text(x_center, y_pos, 'No System 2 / System 3 iterations',
                ha='center', va='center', fontsize=10, fontweight='bold', color='white')
        y_pos -= 1.5

    # Final outcome
    final_info = pipeline_run.get('final_outcome', {})
    final_status = final_info.get('status')
    final_color = success_color if final_status == 'manufacturable' else blocked_color
    final_box = Rectangle((x_center - 1.5, y_pos - 0.5), 3, 0.8,
                          facecolor=final_color, alpha=0.9, edgecolor='black', linewidth=3)
    ax.add_patch(final_box)
    final_candidate = _safe_trunc(final_info.get('final_candidate'), 40, default="N/A")
    ax.text(x_center, y_pos, f'Final Outcome: {_safe_upper(final_status, "UNKNOWN")}\n{final_candidate}',
            ha='center', va='center', fontsize=10, fontweight='bold', color='white')

    ax.set_title('Pipeline Execution Flow', fontsize=16, fontweight='bold', pad=20)

    plt.tight_layout()
    return fig

# Create flow diagram
fig = create_flow_diagram(pipeline_run)
plt.show()

create_flow_diagram(pipeline_run_raw)


### Summary Statistics


In [ ]:
# Display summary statistics
summary_data = {
    'Metric': [
        'Total Execution Time',
        'System 1 Duration',
        'Total System 2 Iterations',
        'Total System 3 Iterations',
        'Candidates Proposed',
        'Candidates Rejected',
        'Final Status',
        'Final Candidate'
    ],
    'Value': [
        f"{pipeline_run.get('total_duration_seconds', 0):.2f} seconds ({pipeline_run.get('total_duration_seconds', 0)/60:.2f} min)",
        f"{pipeline_run.get('system1', {}).get('duration_seconds', 0):.2f} seconds",
        pipeline_run.get('system2_system3_loop', {}).get('total_iterations', 0),
        pipeline_run.get('system2_system3_loop', {}).get('total_iterations', 0),
        pipeline_run.get('system2_system3_loop', {}).get('total_iterations', 0) + pipeline_run.get('final_outcome', {}).get('total_rejected_candidates', 0),
        pipeline_run.get('final_outcome', {}).get('total_rejected_candidates', 0),
        pipeline_run.get('final_outcome', {}).get('status', 'N/A'),
        pipeline_run.get('final_outcome', {}).get('final_candidate', 'N/A')
    ]
}

df_summary = pd.DataFrame(summary_data)
display(df_summary.style.set_properties(**{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#4A90E2'), ('color', 'white'), ('font-weight', 'bold')]},
    {'selector': 'td', 'props': [('padding', '10px')]}
]))

---

## Part 3: Chat & Results Visualization

Inspect agent interactions, KG subgraphs, RAG retrieval, and per-system results.


In [ ]:
def format_timestamp(ts):
    """Format ISO timestamp to readable format"""
    try:
        dt = datetime.fromisoformat(ts.replace('Z', '+00:00'))
        return dt.strftime('%Y-%m-%d %H:%M:%S')
    except:
        return ts

def get_agent_color(agent):
    """Get color for different agents"""
    colors = {
        'research_analyst': '#4A90E2',
        'research_manager': '#50C878',
        'research_scientist': '#FF6B6B',
        'research_assistant': '#9B59B6'
    }
    return colors.get(agent, '#6C757D')

def create_chat_message(role, content, agent=None, timestamp=None, interaction_type=None):
    """Create a styled chat message"""
    if role == 'user':
        bg_color = '#E3F2FD'
        border_color = '#2196F3'
        align = 'right'
        role_name = 'User'
    else:
        bg_color = '#F5F5F5'
        border_color = get_agent_color(agent) if agent else '#757575'
        align = 'left'
        role_name = agent.replace('_', ' ').title() if agent else 'Agent'
    
    type_badge = f'<span style="background: #FFC107; color: #000; padding: 2px 6px; border-radius: 3px; font-size: 10px; margin-left: 8px;">{interaction_type.upper()}</span>' if interaction_type else ''
    
    timestamp_html = f'<div style="font-size: 10px; color: #666; margin-top: 4px;">{format_timestamp(timestamp)}</div>' if timestamp else ''
    
    html = f"""
    <div style="margin: 15px 0; text-align: {align};">
        <div style="
            display: inline-block;
            max-width: 70%;
            background: {bg_color};
            border-left: 4px solid {border_color};
            padding: 12px 16px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            text-align: left;
        ">
            <div style="font-weight: bold; color: {border_color}; margin-bottom: 8px;">
                {role_name}{type_badge}
            </div>
            <div style="color: #333; line-height: 1.6;">
                {content}
            </div>
            {timestamp_html}
        </div>
    </div>
    """
    return html


In [ ]:
def build_subgraph_from_paths(found_paths, kg_name=None):
    """
    Build a NetworkX graph from found_paths data.
    
    Args:
        found_paths: List of path dictionaries from KG query results
        kg_name: Optional filter to only include paths from a specific KG
    
    Returns:
        NetworkX Graph object with nodes and edges from paths
    """
    G = nx.Graph()
    
    for path_info in found_paths:
        # Filter by kg_name if specified
        if kg_name and path_info.get('kg_name') != kg_name:
            continue
        
        # Add nodes with their attributes
        for node in path_info.get('nodes', []):
            node_id = node.get('node_id')
            node_data = node.get('node_data', {})
            if node_id:
                # Add node if not exists, or update attributes
                if not G.has_node(node_id):
                    G.add_node(node_id, **node_data)
                else:
                    # Update attributes if node already exists
                    G.nodes[node_id].update(node_data)
        
        # Add edges with their attributes
        for edge in path_info.get('edges', []):
            source = edge.get('source')
            target = edge.get('target')
            edge_data = edge.get('edge_data', {})
            if source and target:
                # Add edge if not exists, or update attributes
                if not G.has_edge(source, target):
                    G.add_edge(source, target, **edge_data)
                else:
                    # Update attributes if edge already exists
                    G[source][target].update(edge_data)
    
    return G


def visualize_subgraph(G, matched_node_ids=None, title="Knowledge Graph Subgraph", 
                      figsize=(16, 12), node_size_factor=300, font_size=8):
    """
    Visualize a NetworkX subgraph with node coloring and sizing.
    
    Args:
        G: NetworkX graph to visualize
        matched_node_ids: Set/list of node IDs that were matched from keywords (highlighted)
        title: Title for the plot
        figsize: Figure size tuple
        node_size_factor: Multiplier for node sizes
        font_size: Font size for labels
    """
    if len(G.nodes()) == 0:
        print(f"No nodes to visualize for {title}")
        return
    
    plt.figure(figsize=figsize)
    
    # Use spring layout for better node distribution
    pos = nx.spring_layout(G, k=1, iterations=50, seed=42)
    
    # Determine node colors
    node_colors = []
    for node_id in G.nodes():
        node_data = G.nodes[node_id]
        # Use stored color if available
        if 'color' in node_data:
            node_colors.append(node_data['color'])
        # Otherwise use type-based coloring
        elif 'type' in node_data:
            type_colors = {
                'material': '#5796db',
                'property': '#9557db',
                'chemical': '#cc57db',
                'parameter': '#57db96',
                'characterization technique': '#db5796',
                'application': '#db9657'
            }
            node_colors.append(type_colors.get(node_data['type'], '#6C757D'))
        else:
            node_colors.append('#6C757D')
    
    # Determine node sizes
    node_sizes = []
    for node_id in G.nodes():
        node_data = G.nodes[node_id]
        # Use size attribute if available
        if 'size' in node_data:
            size = node_data['size']
        # Otherwise use PageRank
        elif 'pr' in node_data:
            size = max(100, node_data['pr'] * 1000000)  # Scale PageRank
        else:
            size = 300
        node_sizes.append(size)
    
    # Highlight matched nodes (keyword-matched nodes)
    if matched_node_ids:
        matched_set = set(matched_node_ids)
        # Create mapping from node_id to index
        node_list = list(G.nodes())
        node_to_idx = {node_id: idx for idx, node_id in enumerate(node_list)}
        
        node_colors_highlighted = []
        node_sizes_highlighted = []
        for node_id in node_list:
            idx = node_to_idx[node_id]
            if node_id in matched_set:
                # Make matched nodes larger
                node_sizes_highlighted.append(node_sizes[idx] * 1.5)
                node_colors_highlighted.append(node_colors[idx])
            else:
                node_sizes_highlighted.append(node_sizes[idx])
                node_colors_highlighted.append(node_colors[idx])
        node_sizes = node_sizes_highlighted
        node_colors = node_colors_highlighted
    
    # Draw edges first (so they appear behind nodes)
    nx.draw_networkx_edges(G, pos, alpha=0.3, width=0.5, edge_color='gray')
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, 
                           alpha=0.8, edgecolors='black', linewidths=0.5)
    
    # Draw labels for a subset of nodes (to avoid clutter)
    # Label matched nodes and high-degree nodes
    labels = {}
    if matched_node_ids:
        matched_set = set(matched_node_ids)
        for node_id in G.nodes():
            if node_id in matched_set or G.degree(node_id) > 3:
                labels[node_id] = node_id[:30]  # Truncate long labels
    else:
        # Label high-degree nodes
        for node_id in G.nodes():
            if G.degree(node_id) > 3:
                labels[node_id] = node_id[:30]
    
    if labels:
        nx.draw_networkx_labels(G, pos, labels, font_size=font_size, font_weight='bold')
    
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.axis('off')
    plt.tight_layout()
    
    return plt.gcf()


def extract_kg_interactions(chat_log):
    """
    Extract all KG interactions from chat log.
    
    Returns:
        List of KG interaction dictionaries
    """
    kg_interactions = []
    for interaction in chat_log.get('interactions', []):
        if interaction.get('type') == 'kg':
            kg_interactions.append(interaction)
    return kg_interactions

### System 1: Property Extraction Chat Log


In [ ]:
chat_log = pipeline_data['system1']['chat_log']
if chat_log:
    print(f"Pipeline: {chat_log['pipeline']}")
    print(f"Run ID: {chat_log['run_id']}")
    print(f"Interactions: {len(chat_log['interactions'])}")
    print(f"Summary: {chat_log.get('summary', {})}")
else:
    print("System 1 chat log not available")


In [ ]:
# Extract and format chat messages
pipeline = chat_log['pipeline']
run_id = chat_log['run_id']

chat_html = f"""
<style>
    .chat-container {{
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Oxygen, Ubuntu, Cantarell, sans-serif;
        max-width: 900px;
        margin: 0 auto;
        padding: 20px;
        background: #FAFAFA;
        border-radius: 10px;
    }}
    .chat-header {{
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        padding: 20px;
        border-radius: 8px;
        margin-bottom: 20px;
    }}
</style>
<div class="chat-container">
    <div class="chat-header">
        <h2 style="margin: 0;">🤖 Agent Chat Conversation</h2>
        <p style="margin: 5px 0 0 0; opacity: 0.9;">Pipeline: {pipeline} | Run ID: {run_id}</p>
    </div>
"""

# Track the initial user query
initial_query = None

# Process interactions
for i, interaction in enumerate(chat_log['interactions']):
    interaction_type = interaction.get('type', '')
    agent = interaction.get('agent', '')
    timestamp = interaction.get('timestamp', '')
    data = interaction.get('data', {})
    
    if interaction_type == 'rag':
        # RAG queries - show as user queries or agent searches
        query = data.get('query', '')
        if query and not initial_query:
            # First query is the user's initial question
            initial_query = query
            chat_html += create_chat_message(
                'user',
                f"<strong>{query}</strong>",
                timestamp=timestamp,
                interaction_type='RAG Query'
            )
        elif query:
            # Subsequent RAG queries are agent searches
            num_results = data.get('num_results', 0)
            results_summary = f"<em>Found {num_results} relevant documents</em>"
            chat_html += create_chat_message(
                'agent',
                f"<strong>Searching:</strong> {query}<br><br>{results_summary}",
                agent=agent,
                timestamp=timestamp,
                interaction_type='RAG'
            )
    
    elif interaction_type == 'llm':
        # LLM interactions - these are the main conversation
        user_prompt = data.get('user_prompt', '')
        response = data.get('response', '')
        system_prompt = data.get('system_prompt', '')
        
        # Extract the actual user question from the prompt if it contains "Sentence:"
        if 'Sentence:' in user_prompt:
            user_question = user_prompt.split('Sentence:')[1].split('\n')[0].strip()
            if user_question and not initial_query:
                initial_query = user_question
                chat_html += create_chat_message(
                    'user',
                    f"<strong>{user_question}</strong>",
                    timestamp=timestamp,
                    interaction_type='Query'
                )
        
        # Show agent response
        if response:
            # Clean up the response for display
            response_text = response.strip()
            if len(response_text) > 0:
                chat_html += create_chat_message(
                    'agent',
                    response_text.replace('\n', '<br>'),
                    agent=agent,
                    timestamp=timestamp,
                    interaction_type='LLM'
                )
    
    elif interaction_type == 'kg':
        # Knowledge graph interactions
        method = interaction.get('method', '')
        if 'data' in data and data['data']:
            kg_data = data['data']
            if 'keywords' in kg_data:
                keywords = [k.get('keyword', '') for k in kg_data['keywords']]
                if keywords:
                    keywords_text = '<br>'.join([f"• {k}" for k in keywords[:10]])  # Show first 10
                    chat_html += create_chat_message(
                        'agent',
                        f"<strong>Extracted Keywords:</strong><br>{keywords_text}",
                        agent=agent,
                        timestamp=timestamp,
                        interaction_type='KG'
                    )

chat_html += "</div>"
display(HTML(chat_html))


### Knowledge Graph Subgraph Visualization


In [ ]:
# Extract KG interactions from chat log
kg_interactions = extract_kg_interactions(chat_log)

if not kg_interactions:
    print("No KG interactions found in chat log.")
else:
    print(f"Found {len(kg_interactions)} KG interaction(s)\n")
    
    for idx, kg_interaction in enumerate(kg_interactions):
        print(f"{'='*80}")
        print(f"KG Interaction #{idx + 1}")
        print(f"{'='*80}")
        
        data = kg_interaction.get('data', {})
        method = kg_interaction.get('method', 'unknown')
        keywords = data.get('keywords', [])
        found_paths = data.get('found_paths', [])
        result_summary = data.get('result_summary', {})
        matched_node_ids = data.get('matched_node_ids', [])
        
        print(f"Method: {method}")
        print(f"Keywords: {len(keywords)} keywords")
        print(f"Found paths: {len(found_paths)} paths")
        print(f"Matched nodes: {len(matched_node_ids)} nodes")
        
        # Check if multi-KG (separate strategy)
        strategy = result_summary.get('strategy', 'single')
        
        if strategy == 'separate' and 'kg1' in result_summary and 'kg2' in result_summary:
            # Multi-KG mode: visualize each KG separately
            kg1_info = result_summary.get('kg1', {})
            kg2_info = result_summary.get('kg2', {})
            
            kg1_name = kg1_info.get('name', 'kg1')
            kg1_summary = kg1_info.get('summary', {})
            kg1_desc = kg1_info.get('description', '')
            
            kg2_name = kg2_info.get('name', 'kg2')
            kg2_summary = kg2_info.get('summary', {})
            kg2_desc = kg2_info.get('description', '')
            
            print(f"\nMulti-KG mode detected:")
            print(f"  KG1: {kg1_name} - {kg1_summary.get('num_paths_found', 0)} paths, "
                  f"{kg1_summary.get('subgraph_nodes', 0)} nodes, {kg1_summary.get('subgraph_edges', 0)} edges")
            print(f"  KG2: {kg2_name} - {kg2_summary.get('num_paths_found', 0)} paths, "
                  f"{kg2_summary.get('subgraph_nodes', 0)} nodes, {kg2_summary.get('subgraph_edges', 0)} edges")
            
            # Build and visualize KG1 subgraph
            G1 = build_subgraph_from_paths(found_paths, kg_name=kg1_name)
            if len(G1.nodes()) > 0:
                # Filter matched nodes for this KG - nodes that appear in KG1 paths
                kg1_paths = [p for p in found_paths if p.get('kg_name') == kg1_name]
                kg1_path_nodes = set()
                for path_info in kg1_paths:
                    kg1_path_nodes.update(path_info.get('path', []))
                kg1_matched = [nid for nid in matched_node_ids if nid in kg1_path_nodes and nid in G1.nodes()]
                
                visualize_subgraph(
                    G1, 
                    matched_node_ids=kg1_matched if kg1_matched else None,
                    title=f"{kg1_name.upper()} Knowledge Graph\n{kg1_desc}",
                    figsize=(16, 12)
                )
                plt.show()
            else:
                print(f"\nNo nodes found in {kg1_name} subgraph")
            
            # Build and visualize KG2 subgraph
            G2 = build_subgraph_from_paths(found_paths, kg_name=kg2_name)
            if len(G2.nodes()) > 0:
                # Filter matched nodes for this KG - nodes that appear in KG2 paths
                kg2_paths = [p for p in found_paths if p.get('kg_name') == kg2_name]
                kg2_path_nodes = set()
                for path_info in kg2_paths:
                    kg2_path_nodes.update(path_info.get('path', []))
                kg2_matched = [nid for nid in matched_node_ids if nid in kg2_path_nodes and nid in G2.nodes()]
                
                visualize_subgraph(
                    G2,
                    matched_node_ids=kg2_matched if kg2_matched else None,
                    title=f"{kg2_name.upper()} Knowledge Graph\n{kg2_desc}",
                    figsize=(16, 12)
                )
                plt.show()
            else:
                print(f"\nNo nodes found in {kg2_name} subgraph")
        
        else:
            # Single KG mode or merged strategy
            G = build_subgraph_from_paths(found_paths)
            if len(G.nodes()) > 0:
                print(f"\nSingle KG mode:")
                print(f"  Total nodes: {len(G.nodes())}")
                print(f"  Total edges: {len(G.edges())}")
                
                visualize_subgraph(
                    G,
                    matched_node_ids=matched_node_ids if matched_node_ids else None,
                    title=f"Knowledge Graph Subgraph\nMethod: {method}",
                    figsize=(16, 12)
                )
                plt.show()
            else:
                print("\nNo nodes found in subgraph")
        
        # Display summary statistics
        print(f"\nSummary Statistics:")
        print(f"  Algorithm: {result_summary.get('algorithm_used', 'unknown')}")
        print(f"  Connections found: {result_summary.get('connections_found', False)}")
        if 'subgraph_nodes' in result_summary:
            print(f"  Subgraph: {result_summary.get('subgraph_nodes', 0)} nodes, "
                  f"{result_summary.get('subgraph_edges', 0)} edges")
        print()

### RAG Retrieval Results


In [ ]:
# Show RAG results in detail
rag_count = 0
for interaction in chat_log['interactions']:
    if interaction.get('type') == 'rag':
        data = interaction.get('data', {})
        query = data.get('query', '')
        results = data.get('results', [])
        
        if query and results:
            rag_count += 1
            print(f"\n{'='*80}")
            print(f"RAG Query #{rag_count}: {query}")
            print(f"{'='*80}")
            print(f"Found {len(results)} results:\n")
            
            for i, result in enumerate(results[:3], 1):  # Show first 3 results
                doc_id = result.get('id', 'N/A')
                title = result.get('metadata', {}).get('title', 'N/A')
                distance = result.get('distance', 0)
                content = result.get('content', '')[:500]  # First 500 chars
                
                print(f"\n--- Result {i} ---")
                print(f"Title: {title}")
                print(f"ID: {doc_id}")
                print(f"Similarity: {1-distance:.3f}")
                print(f"Content preview: {content}...")
                print()
            
            if len(results) > 3:
                print(f"... and {len(results) - 3} more results\n")


### System 2 ↔ System 3 Iteration Overview


In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle
from matplotlib.patches import Rectangle

def create_iteration_loop_diagram(pipeline_data):
    """
    Create a flow diagram showing the System 2 → System 3 iterative loop with feedback.
    """
    fig, ax = plt.subplots(figsize=(18, 12))
    ax.set_xlim(0, 20)
    ax.set_ylim(0, 15)
    ax.axis('off')
    
    # Colors
    s1_color = '#4A90E2'
    s2_color = '#50C878'
    s3_color = '#FF6B6B'
    success_color = '#2ECC71'
    blocked_color = '#E74C3C'
    feedback_color = '#FF9800'
    
    y_start = 13
    x_center = 10
    
    # System 1 box
    s1_box = FancyBboxPatch((x_center - 2, y_start - 0.6), 4, 1.2,
                            boxstyle="round,pad=0.1", 
                            facecolor=s1_color, edgecolor='black', linewidth=2, alpha=0.9)
    ax.add_patch(s1_box)
    ax.text(x_center, y_start, 'System 1\nProperty Extraction', 
            ha='center', va='center', fontsize=11, fontweight='bold', color='white')
    
    # Arrow down from System 1
    ax.arrow(x_center, y_start - 0.6, 0, -0.8, head_width=0.3, head_length=0.15, 
             fc='black', ec='black', linewidth=2)
    
    y_pos = y_start - 2
    
    # System 2 → System 3 loop
    iterations = pipeline_data['iterations']
    
    for i, iter_data in enumerate(iterations):
        iter_num = iter_data['iteration']
        s2_candidate = iter_data['system2']['candidate']
        s3_status = iter_data['system3']['status']
        
        # Truncate candidate name if too long
        if len(s2_candidate) > 40:
            s2_candidate_display = s2_candidate[:37] + '...'
        else:
            s2_candidate_display = s2_candidate
        
        # System 2 box
        s2_box = FancyBboxPatch((x_center - 2.5, y_pos - 0.6), 5, 1.2,
                                boxstyle="round,pad=0.1",
                                facecolor=s2_color, edgecolor='black', linewidth=2, alpha=0.9)
        ax.add_patch(s2_box)
        ax.text(x_center, y_pos, f'System 2 (Iter {iter_num})\n{s2_candidate_display}',
                ha='center', va='center', fontsize=9, fontweight='bold', color='white')
        
        # Arrow to System 3
        ax.arrow(x_center, y_pos - 0.6, 0, -0.8, head_width=0.3, head_length=0.15,
                 fc='black', ec='black', linewidth=2)
        
        y_pos -= 2
        
        # System 3 box
        s3_box_color = success_color if s3_status == 'manufacturable' else blocked_color
        s3_box = FancyBboxPatch((x_center - 2.5, y_pos - 0.6), 5, 1.2,
                               boxstyle="round,pad=0.1",
                               facecolor=s3_box_color, edgecolor='black', linewidth=2, alpha=0.9)
        ax.add_patch(s3_box)
        status_text = 'MANUFACTURABLE' if s3_status == 'manufacturable' else 'BLOCKED'
        ax.text(x_center, y_pos, f'System 3 (Iter {iter_num})\n{status_text}',
                ha='center', va='center', fontsize=9, fontweight='bold', color='white')
        
        # Feedback arrow (if blocked and not last iteration)
        if s3_status == 'blocked' and i < len(iterations) - 1:
            # Curved arrow from System 3 back to next System 2
            feedback_arrow = FancyArrowPatch(
                (x_center - 2.5, y_pos - 0.6),
                (x_center - 2.5, y_pos + 3.2),
                connectionstyle="arc3,rad=-0.3",
                arrowstyle='->', lw=3, color=feedback_color,
                linestyle='--', alpha=0.8
            )
            ax.add_patch(feedback_arrow)
            
            # Feedback label
            ax.text(x_center - 4.5, y_pos + 1.3, 'Feedback', 
                   rotation=90, ha='center', va='center', 
                   fontsize=10, color=feedback_color, fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=feedback_color, linewidth=2))
        
        y_pos -= 2
    
    # Final outcome box
    final_status = pipeline_data['final_outcome'].get('status', 'unknown')
    final_color = success_color if final_status == 'manufacturable' else blocked_color
    final_box = FancyBboxPatch((x_center - 3, y_pos - 0.6), 6, 1.2,
                              boxstyle="round,pad=0.1",
                              facecolor=final_color, edgecolor='black', linewidth=3, alpha=0.95)
    ax.add_patch(final_box)
    final_candidate = pipeline_data['final_outcome'].get('final_candidate', 'N/A')
    if final_candidate and len(final_candidate) > 50:
        final_candidate = final_candidate[:47] + '...'
    ax.text(x_center, y_pos, f'Final Outcome: {final_status.upper()}\n{final_candidate}',
            ha='center', va='center', fontsize=10, fontweight='bold', color='white')
    
    ax.set_title('Hierarchical Iterative System: System 2 ↔ System 3 Loop', 
                fontsize=16, fontweight='bold', pad=20)
    
    plt.tight_layout()
    return fig

# Create iteration loop diagram
if 'pipeline_data' in globals() and pipeline_data['iterations']:
    fig = create_iteration_loop_diagram(pipeline_data)
    plt.show()
else:
    print("No iteration data available to visualize.")

In [ ]:
def create_iteration_summary_cards(pipeline_data):
    """Create HTML summary cards for each iteration"""
    iterations = pipeline_data['iterations']
    
    html = f"""
    <div style="font-family: Arial, sans-serif; padding: 20px;">
        <h2 style="color: #333; margin-top: 0;">Iteration Summary</h2>
        <div style="display: grid; grid-template-columns: repeat({min(len(iterations), 3)}, 1fr); gap: 20px;">
    """
    
    for iter_data in iterations:
        iter_num = iter_data['iteration']
        s2_data = iter_data['system2']
        s3_data = iter_data['system3']
        
        candidate = s2_data['candidate']
        if len(candidate) > 60:
            candidate = candidate[:57] + '...'
        
        status = s3_data['status']
        status_color = '#2ECC71' if status == 'manufacturable' else '#E74C3C'
        status_text = '✓ Manufacturable' if status == 'manufacturable' else '✗ Blocked'
        
        num_constraints = len(s3_data.get('blocking_constraints', []))
        feedback = s3_data.get('feedback_to_system2', '')
        if feedback and len(feedback) > 100:
            feedback = feedback[:97] + '...'
        
        html += f"""
        <div style="
            background: white;
            border: 2px solid {status_color};
            border-radius: 10px;
            padding: 15px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
        ">
            <h3 style="color: {status_color}; margin-top: 0; border-bottom: 2px solid {status_color}; padding-bottom: 8px;">
                Iteration {iter_num}
            </h3>
            <div style="margin-bottom: 10px;">
                <strong>Candidate:</strong><br>
                <div style="color: #666; font-size: 14px; margin-top: 4px;">{candidate}</div>
            </div>
            <div style="margin-bottom: 10px;">
                <strong>System 3 Status:</strong><br>
                <span style="color: {status_color}; font-weight: bold;">{status_text}</span>
            </div>
            <div style="margin-bottom: 10px;">
                <strong>Duration:</strong><br>
                <div style="color: #666; font-size: 13px;">
                    S2: {s2_data['duration_seconds']:.1f}s | S3: {s3_data['duration_seconds']:.1f}s
                </div>
            </div>
        """
        
        if status == 'blocked':
            html += f"""
            <div style="margin-top: 10px; padding: 8px; background: #FFF5F5; border-radius: 5px;">
                <strong>Blocking Constraints:</strong> {num_constraints}
            </div>
            """
            if feedback:
                html += f"""
                <div style="margin-top: 10px; padding: 8px; background: #FFF9E6; border-radius: 5px; font-size: 12px;">
                    <strong>Feedback:</strong> {feedback}
                </div>
                """
        
        html += "</div>"
    
    html += """
        </div>
    </div>
    """
    
    return html

if 'pipeline_data' in globals() and pipeline_data['iterations']:
    display(HTML(create_iteration_summary_cards(pipeline_data)))

create_iteration_summary_cards(pipeline_data)


### Feedback Flow Analysis


In [ ]:
def visualize_feedback_flow(pipeline_data):
    """Visualize feedback flow from System 3 to System 2 across iterations"""
    iterations = pipeline_data['iterations']
    
    html = f"""
    <div style="font-family: Arial, sans-serif; padding: 20px; background: #FAFAFA; border-radius: 10px;">
        <h2 style="color: #333; margin-top: 0;">Feedback Flow Across Iterations</h2>
    """
    
    for i, iter_data in enumerate(iterations):
        iter_num = iter_data['iteration']
        s2_data = iter_data['system2']
        s3_data = iter_data['system3']
        
        feedback = s3_data.get('feedback_to_system2', '')
        blocking_constraints = s3_data.get('blocking_constraints', [])
        
        if feedback or blocking_constraints:
            html += f"""
            <div style="
                background: white;
                border-left: 4px solid #FF9800;
                padding: 20px;
                border-radius: 8px;
                margin-bottom: 20px;
                box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            ">
                <h3 style="color: #FF9800; margin-top: 0;">
                    Iteration {iter_num} → Iteration {iter_num + 1} Feedback
                </h3>
                <div style="margin-bottom: 15px;">
                    <strong>From System 3 (Iter {iter_num}):</strong>
                    <div style="padding: 10px; background: #FFF5F5; border-radius: 5px; margin-top: 8px;">
                        <div style="color: #666; line-height: 1.6;">{feedback if feedback else 'No explicit feedback provided'}</div>
                    </div>
                </div>
            """
            
            if blocking_constraints:
                html += f"""
                <div style="margin-top: 15px;">
                    <strong>Blocking Constraints ({len(blocking_constraints)}):</strong>
                    <ul style="margin: 8px 0; padding-left: 20px;">
                """
                for constraint in blocking_constraints[:3]:  # Show first 3
                    constraint_type = constraint.get('type', 'unknown')
                    severity = constraint.get('severity', 'hard')
                    description = constraint.get('description', '')
                    if len(description) > 150:
                        description = description[:147] + '...'
                    
                    severity_color = '#FF4444' if severity == 'hard' else '#FF8800'
                    html += f"""
                    <li style="margin-bottom: 8px;">
                        <span style="color: {severity_color}; font-weight: bold;">[{severity.upper()}]</span>
                        <span style="color: #666;">{constraint_type.replace('_', ' ').title()}: {description}</span>
                    </li>
                    """
                if len(blocking_constraints) > 3:
                    html += f'<li style="color: #999;">... and {len(blocking_constraints) - 3} more constraints</li>'
                html += "</ul></div>"
            
            # Show how feedback is incorporated (if next iteration exists)
            if i < len(iterations) - 1:
                next_iter = iterations[i + 1]
                next_s2_result = next_iter['system2']['result']
                if next_s2_result:
                    constraints = next_s2_result.get('constraints', [])
                    # Find constraints that mention feedback
                    feedback_constraints = [c for c in constraints if 'System 3 feedback' in c or 'feedback' in c.lower()]
                    
                    if feedback_constraints:
                        html += f"""
                        <div style="margin-top: 15px; padding: 10px; background: #E8F5E9; border-radius: 5px;">
                            <strong>Incorporated into System 2 (Iter {iter_num + 1}) Constraints:</strong>
                            <ul style="margin: 8px 0; padding-left: 20px;">
                        """
                        for fc in feedback_constraints[:2]:  # Show first 2
                            if len(fc) > 200:
                                fc = fc[:197] + '...'
                            html += f'<li style="color: #666; margin-bottom: 5px;">{fc}</li>'
                        html += "</ul></div>"
            
            html += "</div>"
    
    html += "</div>"
    return html

if 'pipeline_data' in globals() and pipeline_data['iterations']:
    display(HTML(visualize_feedback_flow(pipeline_data)))

visualize_feedback_flow(pipeline_data)


### System 2: Material Discovery


In [ ]:
# Visualize System 2 iterations
if 'pipeline_data' in globals() and pipeline_data['iterations']:
    for iter_data in pipeline_data['iterations']:
        iter_num = iter_data['iteration']
        system2_chat_log = iter_data['system2']['chat_log']
        system2_result = iter_data['system2']['result']
        
        if not system2_chat_log:
            print(f"Skipping Iteration {iter_num}: System 2 chat log not available")
            continue
        
        # Display iteration header
        display(HTML(f"""
        <div style="
            background: linear-gradient(135deg, #50C878 0%, #4A90E2 100%);
            color: white;
            padding: 20px;
            border-radius: 8px;
            margin: 30px 0 20px 0;
        ">
            <h2 style="margin: 0;">🔬 System 2: Material Discovery - Iteration {iter_num}</h2>
            <p style="margin: 5px 0 0 0; opacity: 0.9;">
                Run ID: {iter_data['system2']['run_id']} | 
                Candidate: {iter_data['system2']['candidate'][:80]}{'...' if len(iter_data['system2']['candidate']) > 80 else ''}
            </p>
        </div>
        """))
        
        # Show constraints including feedback from previous iteration
        if system2_result:
            constraints = system2_result.get('constraints', [])
            feedback_constraints = [c for c in constraints if 'System 3 feedback' in c or 'feedback' in c.lower()]
            if feedback_constraints:
                display(HTML(f"""
            <div style="
                background: #FFF9E6;
                border-left: 4px solid #FF9800;
                padding: 15px;
                border-radius: 8px;
                margin-bottom: 20px;
            ">
                <h4 style="margin: 0 0 10px 0; color: #FF9800;">📝 Feedback Incorporated from Previous Iteration:</h4>
                <ul style="margin: 5px 0; padding-left: 20px;">
        """))
                for fc in feedback_constraints[:3]:
                    if len(fc) > 200:
                        fc = fc[:197] + '...'
                    display(HTML(f'<li style="color: #666; margin-bottom: 5px;">{fc}</li>'))
                display(HTML("</ul></div>"))
        
        # Visualize System 2 chat log
        if system2_chat_log:
            pipeline = system2_chat_log['pipeline']
            run_id = system2_chat_log['run_id']
            
            chat_html = f"""
            <style>
                .chat-container {{
                    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Oxygen, Ubuntu, Cantarell, sans-serif;
                    max-width: 900px;
                    margin: 0 auto;
                    padding: 20px;
                    background: #FAFAFA;
                    border-radius: 10px;
                }}
                .chat-header {{
                    background: linear-gradient(135deg, #50C878 0%, #4A90E2 100%);
                    color: white;
                    padding: 20px;
                    border-radius: 8px;
                    margin-bottom: 20px;
                }}
            </style>
            <div class="chat-container">
                <div class="chat-header">
                    <h2 style="margin: 0;">🔬 System 2: Material Discovery Chat</h2>
                    <p style="margin: 5px 0 0 0; opacity: 0.9;">Pipeline: {pipeline} | Run ID: {run_id}</p>
                </div>
            """
            
            # Track the initial user query
            initial_query = None
            
            # Process interactions
            for i, interaction in enumerate(system2_chat_log['interactions']):
                interaction_type = interaction.get('type', '')
                agent = interaction.get('agent', '')
                timestamp = interaction.get('timestamp', '')
                data = interaction.get('data', {})
                
                if interaction_type == 'rag':
                    # RAG queries - show as user queries or agent searches
                    query = data.get('query', '')
                    if query:
                        num_results = data.get('num_results', 0)
                        results_summary = f"<em>Found {num_results} relevant documents</em>"
                        chat_html += create_chat_message(
                            'agent',
                            f"<strong>Searching:</strong> {query}<br><br>{results_summary}",
                            agent=agent,
                            timestamp=timestamp,
                            interaction_type='RAG'
                        )
                
                elif interaction_type == 'llm':
                    # LLM interactions - these are the main conversation
                    user_prompt = data.get('user_prompt', '')
                    response = data.get('response', '')
                    method = interaction.get('method', '')
                    
                    # Show user prompt for certain methods
                    if method in ['propose_candidate', 'generate_validation_queries', 'validate_feasibility']:
                        if user_prompt and not initial_query:
                            # Extract key information from prompt
                            if 'Material to replace' in user_prompt:
                                initial_query = "Material Discovery Request"
                                chat_html += create_chat_message(
                                    'user',
                                    f"<strong>Material Discovery Request</strong>",
                                    timestamp=timestamp,
                                    interaction_type='Query'
                                )
                    
                    # Show agent response
                    if response:
                        response_text = response.strip()
                        if len(response_text) > 0:
                            # Show full response (no truncation)
                            chat_html += create_chat_message(
                                'agent',
                                response_text.replace('\n', '<br>'),
                                agent=agent,
                                timestamp=timestamp,
                                interaction_type=f'LLM ({method})'
                            )
                
                elif interaction_type == 'kg':
                    # Knowledge graph interactions
                    method = interaction.get('method', '')
                    if 'data' in data and data['data']:
                        kg_data = data['data']
                        if 'keywords' in kg_data:
                            keywords = [k.get('keyword', '') for k in kg_data['keywords']]
                            if keywords:
                                keywords_text = '<br>'.join([f"• {k}" for k in keywords[:10]])  # Show first 10
                                chat_html += create_chat_message(
                                    'agent',
                                    f"<strong>Extracted Keywords:</strong><br>{keywords_text}",
                                    agent=agent,
                                    timestamp=timestamp,
                                    interaction_type='KG'
                                )
            
            chat_html += "</div>"
            display(HTML(chat_html))
        
        # Add separator between iterations
        if iter_num < len(pipeline_data['iterations']):
            display(HTML('<hr style="margin: 40px 0; border: 2px solid #ddd;">'))
else:
    print("No System 2 iterations available to visualize.")

### System 2: Discovery Results


In [ ]:
def create_material_discovery_summary(discovery_results):
    """Create HTML summary card for material discovery results"""
    if not discovery_results:
        return ""
    
    success = discovery_results.get('success', False)
    candidate = discovery_results.get('candidate', {})
    iterations = discovery_results.get('iterations', 0)
    rejected_count = len(discovery_results.get('rejected_candidates', []))
    
    status_color = '#50C878' if success else '#FF6B6B'
    status_text = '✓ SUCCESS' if success else '✗ NO CANDIDATE FOUND'
    
    candidate_name = candidate.get('material_name', 'N/A') if success else 'N/A'
    
    html = f"""
    <div style="
        background: linear-gradient(135deg, {status_color} 0%, #4A90E2 100%);
        color: white;
        padding: 20px;
        border-radius: 8px;
        margin-bottom: 20px;
    ">
        <h2 style="margin: 0 0 10px 0;">🔬 Material Discovery Results</h2>
        <div style="display: grid; grid-template-columns: repeat(2, 1fr); gap: 15px; margin-top: 15px;">
            <div>
                <div style="font-size: 12px; opacity: 0.9;">Status</div>
                <div style="font-size: 20px; font-weight: bold;">{status_text}</div>
            </div>
            <div>
                <div style="font-size: 12px; opacity: 0.9;">Candidate Material</div>
                <div style="font-size: 20px; font-weight: bold;">{candidate_name}</div>
            </div>
            <div>
                <div style="font-size: 12px; opacity: 0.9;">Iterations</div>
                <div style="font-size: 20px; font-weight: bold;">{iterations}</div>
            </div>
            <div>
                <div style="font-size: 12px; opacity: 0.9;">Rejected Candidates</div>
                <div style="font-size: 20px; font-weight: bold;">{rejected_count}</div>
            </div>
        </div>
    </div>
    """
    return html

if discovery_results:
    display(HTML(create_material_discovery_summary(discovery_results)))

In [ ]:
def visualize_candidate_details(discovery_results):
    """Display successful candidate details and justification"""
    if not discovery_results or not discovery_results.get('success'):
        print("No successful candidate found")
        return
    
    candidate = discovery_results.get('candidate', {})
    material_name = candidate.get('material_name', 'Unknown')
    justification = candidate.get('justification', '')
    
    html = f"""
    <div style="
        background: #F5F5F5;
        border-left: 4px solid #50C878;
        padding: 20px;
        border-radius: 8px;
        margin: 20px 0;
    ">
        <h3 style="margin: 0 0 15px 0; color: #50C878;">✓ Successful Candidate: {material_name}</h3>
        <div style="margin-top: 15px;">
            <strong>Justification:</strong>
            <div style="margin-top: 10px; padding: 15px; background: white; border-radius: 5px; line-height: 1.6;">
                {justification.replace(chr(10), '<br>')}
            </div>
        </div>
    </div>
    """
    display(HTML(html))

if discovery_results:
    visualize_candidate_details(discovery_results)

In [ ]:
def visualize_rejected_candidates(discovery_results):
    """Display rejected candidates with reasons"""
    if not discovery_results:
        return
    
    rejected_candidates = discovery_results.get('rejected_candidates', [])
    iteration_history = discovery_results.get('iteration_history', [])
    
    if not rejected_candidates:
        print("No rejected candidates")
        return
    
    # Build a map of candidate to rejection reasons
    rejection_map = {}
    for hist in iteration_history:
        if not hist.get('feasible', False):
            candidate = hist.get('candidate', 'Unknown')
            constraints = hist.get('constraints_violated', [])
            reasoning = hist.get('reasoning', '')
            rejection_map[candidate] = {
                'constraints': constraints,
                'reasoning': reasoning,
                'iteration': hist.get('iteration', 0)
            }
    
    html = f"""
    <div style="
        background: #FFF5F5;
        border-left: 4px solid #FF6B6B;
        padding: 20px;
        border-radius: 8px;
        margin: 20px 0;
    ">
        <h3 style="margin: 0 0 15px 0; color: #FF6B6B;">✗ Rejected Candidates ({len(rejected_candidates)})</h3>
    """
    
    for candidate in rejected_candidates:
        rejection_info = rejection_map.get(candidate, {})
        constraints = rejection_info.get('constraints', [])
        reasoning = rejection_info.get('reasoning', '')
        iteration = rejection_info.get('iteration', 0)
        
        html += f"""
        <div style="
            background: white;
            padding: 15px;
            border-radius: 5px;
            margin-bottom: 10px;
            border-left: 3px solid #FF6B6B;
        ">
            <div style="font-weight: bold; font-size: 16px; color: #FF6B6B; margin-bottom: 8px;">
                {candidate} (Iteration {iteration})
            </div>
        """
        
        if constraints:
            html += f"""
            <div style="margin-top: 8px;">
                <strong>Constraints Violated:</strong>
                <ul style="margin: 5px 0;">
            """
            for constraint in constraints[:3]:  # Show first 3
                html += f"<li>{constraint}</li>"
            html += "</ul></div>"
        
        if reasoning:
            html += f"""
            <div style="margin-top: 8px; font-style: italic; color: #666;">
                {reasoning}
            </div>
            """
        
        html += "</div>"
    
    html += "</div>"
    display(HTML(html))

if discovery_results:
    visualize_rejected_candidates(discovery_results)

In [ ]:
# Load System 2 material discovery results using auto-detected path
if 'system2_result_path' not in globals() or not os.path.exists(system2_result_path):
    print("No System 2 material discovery results found. Run System 2 pipeline first.")
    discovery_results = None
else:
    print(f"Loading System 2 material discovery results from: {system2_result_path}")
    with open(system2_result_path, 'r') as f:
        discovery_results = json.load(f)
    print(f"Run ID: {discovery_results.get('run_id', 'N/A')}")
if 'system2_results' in dir() and system2_results:
    create_material_discovery_summary(system2_results)
    visualize_candidate_details(system2_results)
    visualize_rejected_candidates(system2_results)


### System 3: Manufacturability Assessment


In [ ]:
# Use auto-detected System 3 chat log path from initialization
if 'system3_chat_log_path' not in globals() or system3_chat_log_path is None or not os.path.exists(system3_chat_log_path):
    print("System 3 chat log not found. Set system3_chat_log_path or run System 3 pipeline first.")
    system3_chat_log = None
else:
    print(f"Loading System 3 chat log: {system3_chat_log_path}")
    # Load System 3 chat log
    with open(system3_chat_log_path, 'r') as f:
        system3_chat_log = json.load(f)
    
    print(f"Pipeline: {system3_chat_log['pipeline']}")
    print(f"Run ID: {system3_chat_log['run_id']}")
    print(f"Start Time: {system3_chat_log['timestamp']}")
    print(f"End Time: {system3_chat_log.get('end_time', 'N/A')}")
    print(f"Duration: {system3_chat_log.get('duration_seconds', 0):.2f} seconds")
    print(f"\nTotal Interactions: {len(system3_chat_log['interactions'])}")
    print(f"Summary: {system3_chat_log.get('summary', {})}")

In [ ]:
# Visualize System 3 chat log (if available)
if system3_chat_log:
    pipeline = system3_chat_log['pipeline']
    run_id = system3_chat_log['run_id']
    
    chat_html = f"""
    <style>
        .chat-container {{
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Oxygen, Ubuntu, Cantarell, sans-serif;
            max-width: 900px;
            margin: 0 auto;
            padding: 20px;
            background: #FAFAFA;
            border-radius: 10px;
        }}
        .chat-header {{
            background: linear-gradient(135deg, #FF6B35 0%, #9B59B6 100%);
            color: white;
            padding: 20px;
            border-radius: 8px;
            margin-bottom: 20px;
        }}
    </style>
    <div class="chat-container">
        <div class="chat-header">
            <h2 style="margin: 0;">🏭 System 3: Manufacturability Assessment Chat</h2>
            <p style="margin: 5px 0 0 0; opacity: 0.9;">Pipeline: {pipeline} | Run ID: {run_id}</p>
        </div>
    """
    
    # Track the initial user query
    initial_query = None
    
    # Process interactions
    for i, interaction in enumerate(system3_chat_log['interactions']):
        interaction_type = interaction.get('type', '')
        agent = interaction.get('agent', '')
        timestamp = interaction.get('timestamp', '')
        data = interaction.get('data', {})
        method = interaction.get('method', '')
        
        if interaction_type == 'rag':
            # RAG queries - show as agent searches
            query = data.get('query', '')
            if query:
                num_results = data.get('num_results', 0)
                results_summary = f"<em>Found {num_results} relevant documents</em>"
                chat_html += create_chat_message(
                    'agent',
                    f"<strong>Searching:</strong> {query}<br><br>{results_summary}",
                    agent=agent,
                    timestamp=timestamp,
                    interaction_type='RAG'
                )
        
        elif interaction_type == 'llm':
            # LLM interactions - these are the main conversation
            user_prompt = data.get('user_prompt', '')
            response = data.get('response', '')
            
            # Show user prompt for certain methods
            if method in ['generate_process_retrieval_queries', 'assess_manufacturability_feasibility', 'synthesize_process_recipe']:
                if user_prompt and not initial_query:
                    # Extract key information from prompt
                    if 'Candidate material:' in user_prompt:
                        material_match = user_prompt.split('Candidate material:')[1].split('\n')[0].strip()
                        initial_query = f"Manufacturability Assessment for {material_match}"
                        chat_html += create_chat_message(
                            'user',
                            f"<strong>{initial_query}</strong>",
                            timestamp=timestamp,
                            interaction_type='Query'
                        )
            
            # Show agent response
            if response:
                response_text = response.strip()
                if len(response_text) > 0:
                    # Show full response (no truncation)
                    chat_html += create_chat_message(
                        'agent',
                        response_text.replace('\n', '<br>'),
                        agent=agent,
                        timestamp=timestamp,
                        interaction_type=f'LLM ({method})'
                    )
        
        elif interaction_type == 'kg':
            # Knowledge graph interactions
            method = interaction.get('method', '')
            if 'data' in data and data['data']:
                kg_data = data['data']
                if 'keywords' in kg_data:
                    keywords = [k.get('keyword', '') for k in kg_data['keywords']]
                    if keywords:
                        keywords_text = '<br>'.join([f"• {k}" for k in keywords[:10]])  # Show first 10
                        chat_html += create_chat_message(
                            'agent',
                            f"<strong>Extracted Keywords:</strong><br>{keywords_text}",
                            agent=agent,
                            timestamp=timestamp,
                            interaction_type='KG'
                        )
    
    chat_html += "</div>"
    display(HTML(chat_html))
else:
    print("No System 3 chat log available to visualize.")

### System 3: Assessment Results


In [ ]:
def create_system3_summary(system3_results):
    """Create HTML summary card for System 3 manufacturability assessment results"""
    if not system3_results:
        return ""
    
    status = system3_results.get('status', 'unknown')
    candidate = system3_results.get('candidate', {})
    
    candidate_name = candidate.get('material_name', 'N/A') if candidate else 'N/A'
    
    if status == 'manufacturable':
        status_color = '#50C878'
        status_text = '✓ MANUFACTURABLE'
        process_recipe = system3_results.get('process_recipe', [])
        num_steps = len(process_recipe)
        metric_label = 'Process Steps'
        metric_value = num_steps
    else:
        status_color = '#FF6B6B'
        status_text = '✗ BLOCKED'
        blocking_constraints = system3_results.get('blocking_constraints', [])
        num_constraints = len(blocking_constraints)
        metric_label = 'Blocking Constraints'
        metric_value = num_constraints
    
    html = f"""
    <div style="
        background: linear-gradient(135deg, {status_color} 0%, #9B59B6 100%);
        color: white;
        padding: 20px;
        border-radius: 8px;
        margin-bottom: 20px;
    ">
        <h2 style="margin: 0 0 10px 0;">🏭 Manufacturability Assessment Results</h2>
        <div style="display: grid; grid-template-columns: repeat(2, 1fr); gap: 15px; margin-top: 15px;">
            <div>
                <div style="font-size: 12px; opacity: 0.9;">Status</div>
                <div style="font-size: 20px; font-weight: bold;">{status_text}</div>
            </div>
            <div>
                <div style="font-size: 12px; opacity: 0.9;">Candidate Material</div>
                <div style="font-size: 20px; font-weight: bold;">{candidate_name}</div>
            </div>
            <div>
                <div style="font-size: 12px; opacity: 0.9;">{metric_label}</div>
                <div style="font-size: 20px; font-weight: bold;">{metric_value}</div>
            </div>
        </div>
    </div>
    """
    return html

if system3_results:
    display(HTML(create_system3_summary(system3_results)))

In [ ]:
def visualize_process_recipe(system3_results):
    """Display process recipe for manufacturable materials"""
    if not system3_results or system3_results.get('status') != 'manufacturable':
        return
    
    process_recipe = system3_results.get('process_recipe', [])
    candidate = system3_results.get('candidate', {})
    material_name = candidate.get('material_name', 'Unknown') if candidate else 'Unknown'
    evidence = system3_results.get('evidence', [])
    
    if not process_recipe:
        print("No process recipe available")
        return
    
    html = f"""
    <div style="
        background: #F5F5F5;
        border-left: 4px solid #50C878;
        padding: 20px;
        border-radius: 8px;
        margin: 20px 0;
    ">
        <h3 style="margin: 0 0 15px 0; color: #50C878;">✓ Process Recipe: {material_name}</h3>
    """
    
    for step in process_recipe:
        step_index = step.get('step_index', 0)
        description = step.get('description', '')
        conditions = step.get('conditions', '')
        equipment = step.get('equipment_class', '')
        inputs = step.get('inputs', [])
        
        html += f"""
        <div style="
            background: white;
            padding: 15px;
            border-radius: 5px;
            margin-bottom: 10px;
            border-left: 3px solid #50C878;
        ">
            <div style="font-weight: bold; font-size: 16px; color: #50C878; margin-bottom: 8px;">
                Step {step_index}: {description}
            </div>
        """
        
        if conditions:
            html += f"""
            <div style="margin-top: 8px;">
                <strong>Conditions:</strong> {conditions}
            </div>
            """
        
        if equipment:
            html += f"""
            <div style="margin-top: 8px;">
                <strong>Equipment:</strong> {equipment}
            </div>
            """
        
        if inputs:
            inputs_str = ', '.join(inputs) if isinstance(inputs, list) else str(inputs)
            html += f"""
            <div style="margin-top: 8px;">
                <strong>Inputs:</strong> {inputs_str}
            </div>
            """
        
        html += "</div>"
    
    if evidence:
        evidence_str = ', '.join([str(e) for e in evidence[:5]])  # Show first 5
        if len(evidence) > 5:
            evidence_str += f" ... and {len(evidence) - 5} more"
        html += f"""
        <div style="margin-top: 15px; padding: 10px; background: #E8F5E9; border-radius: 5px;">
            <strong>Evidence:</strong> {evidence_str}
        </div>
        """
    
    html += "</div>"
    display(HTML(html))

if system3_results and system3_results.get('status') == 'manufacturable':
    visualize_process_recipe(system3_results)

In [ ]:
# Load System 3 manufacturability assessment results using auto-detected path
if 'system3_result_path' not in globals() or system3_result_path is None or not os.path.exists(system3_result_path):
    print("No System 3 results found. Run System 3 pipeline first.")
    system3_results = None
else:
    print(f"Loading System 3 results from: {system3_result_path}")
    with open(system3_result_path, 'r') as f:
        system3_results = json.load(f)
    print(f"Run ID: {system3_results.get('run_id', 'N/A')}")
    print(f"Status: {system3_results.get('status', 'N/A')}")
if 'system3_results' in dir() and system3_results:
    create_system3_summary(system3_results)
    visualize_process_recipe(system3_results)


### Constraint Evolution


In [ ]:
def visualize_constraint_evolution(pipeline_data):
    """Visualize how constraints evolve across iterations"""
    iterations = pipeline_data['iterations']
    
    # Collect constraint data
    constraint_data = []
    for iter_data in iterations:
        iter_num = iter_data['iteration']
        s2_result = iter_data['system2']['result']
        s3_result = iter_data['system3']['result']
        
        if s2_result:
            constraints = s2_result.get('constraints', [])
            # Count different types
            total_constraints = len(constraints)
            feedback_constraints = len([c for c in constraints if 'System 3 feedback' in c or 'feedback' in c.lower()])
            hard_constraints = len([c for c in constraints if 'Hard' in c or 'hard constraint' in c.lower()])
            
            constraint_data.append({
                'iteration': iter_num,
                'total': total_constraints,
                'feedback_based': feedback_constraints,
                'hard': hard_constraints,
                'candidate': iter_data['system2']['candidate'][:50] + '...' if len(iter_data['system2']['candidate']) > 50 else iter_data['system2']['candidate']
            })
    
    if not constraint_data:
        print("No constraint data available")
        return
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    iterations_list = [d['iteration'] for d in constraint_data]
    totals = [d['total'] for d in constraint_data]
    feedback_based = [d['feedback_based'] for d in constraint_data]
    hard = [d['hard'] for d in constraint_data]
    
    # Bar chart
    x = np.arange(len(iterations_list))
    width = 0.35
    
    ax1.bar(x - width/2, totals, width, label='Total Constraints', color='#4A90E2', alpha=0.8)
    ax1.bar(x + width/2, feedback_based, width, label='Feedback-Based', color='#FF9800', alpha=0.8)
    
    ax1.set_xlabel('Iteration', fontsize=12)
    ax1.set_ylabel('Number of Constraints', fontsize=12)
    ax1.set_title('Constraint Count Evolution', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels([f"Iter {i}" for i in iterations_list])
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Line chart showing accumulation
    ax2.plot(iterations_list, totals, marker='o', linewidth=2, markersize=8, label='Total', color='#4A90E2')
    ax2.plot(iterations_list, feedback_based, marker='s', linewidth=2, markersize=8, label='Feedback-Based', color='#FF9800')
    ax2.plot(iterations_list, hard, marker='^', linewidth=2, markersize=8, label='Hard', color='#E74C3C')
    
    ax2.set_xlabel('Iteration', fontsize=12)
    ax2.set_ylabel('Number of Constraints', fontsize=12)
    ax2.set_title('Constraint Accumulation Trend', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    return fig

if 'pipeline_data' in globals() and pipeline_data['iterations']:
    fig = visualize_constraint_evolution(pipeline_data)
    plt.show()

visualize_constraint_evolution(pipeline_data)


### Iteration Comparison


In [ ]:
def create_iteration_comparison(pipeline_data):
    """Create side-by-side comparison of iterations"""
    iterations = pipeline_data['iterations']
    
    html = f"""
    <div style="font-family: Arial, sans-serif; padding: 20px;">
        <h2 style="color: #333; margin-top: 0;">Iteration Comparison</h2>
        <div style="overflow-x: auto;">
            <table style="width: 100%; border-collapse: collapse; margin-top: 20px;">
                <thead>
                    <tr style="background: #4A90E2; color: white;">
                        <th style="padding: 12px; text-align: left; border: 1px solid #ddd;">Iteration</th>
                        <th style="padding: 12px; text-align: left; border: 1px solid #ddd;">Candidate Material</th>
                        <th style="padding: 12px; text-align: left; border: 1px solid #ddd;">S2 Duration</th>
                        <th style="padding: 12px; text-align: left; border: 1px solid #ddd;">S3 Status</th>
                        <th style="padding: 12px; text-align: left; border: 1px solid #ddd;">S3 Duration</th>
                        <th style="padding: 12px; text-align: left; border: 1px solid #ddd;">Blocking Constraints</th>
                        <th style="padding: 12px; text-align: left; border: 1px solid #ddd;">Feedback Provided</th>
                    </tr>
                </thead>
                <tbody>
    """
    
    for iter_data in iterations:
        iter_num = iter_data['iteration']
        s2_data = iter_data['system2']
        s3_data = iter_data['system3']
        
        candidate = s2_data['candidate']
        if len(candidate) > 60:
            candidate = candidate[:57] + '...'
        
        status = s3_data['status']
        status_color = '#2ECC71' if status == 'manufacturable' else '#E74C3C'
        status_text = '✓ Manufacturable' if status == 'manufacturable' else '✗ Blocked'
        
        num_constraints = len(s3_data.get('blocking_constraints', []))
        feedback = s3_data.get('feedback_to_system2', '')
        has_feedback = '✓' if feedback else '✗'
        
        html += f"""
                    <tr style="background: {'#f9f9f9' if iter_num % 2 == 0 else 'white'};">
                        <td style="padding: 10px; border: 1px solid #ddd; font-weight: bold;">{iter_num}</td>
                        <td style="padding: 10px; border: 1px solid #ddd;">{candidate}</td>
                        <td style="padding: 10px; border: 1px solid #ddd;">{s2_data['duration_seconds']:.1f}s</td>
                        <td style="padding: 10px; border: 1px solid #ddd; color: {status_color}; font-weight: bold;">{status_text}</td>
                        <td style="padding: 10px; border: 1px solid #ddd;">{s3_data['duration_seconds']:.1f}s</td>
                        <td style="padding: 10px; border: 1px solid #ddd;">{num_constraints}</td>
                        <td style="padding: 10px; border: 1px solid #ddd;">{has_feedback}</td>
                    </tr>
        """
    
    html += """
                </tbody>
            </table>
        </div>
    </div>
    """
    
    return html

if 'pipeline_data' in globals() and pipeline_data['iterations']:
    display(HTML(create_iteration_comparison(pipeline_data)))

create_iteration_comparison(pipeline_data)


In [ ]:
def visualize_blocking_constraints(system3_results):
    """Display blocking constraints for blocked materials"""
    if not system3_results or system3_results.get('status') != 'blocked':
        return
    
    blocking_constraints = system3_results.get('blocking_constraints', [])
    candidate = system3_results.get('candidate', {})
    material_name = candidate.get('material_name', 'Unknown') if candidate else 'Unknown'
    feedback_to_system2 = system3_results.get('feedback_to_system2', '')
    
    if not blocking_constraints:
        print("No blocking constraints available")
        return
    
    html = f"""
    <div style="
        background: #FFF5F5;
        border-left: 4px solid #FF6B6B;
        padding: 20px;
        border-radius: 8px;
        margin: 20px 0;
    ">
        <h3 style="margin: 0 0 15px 0; color: #FF6B6B;">✗ Blocking Constraints: {material_name}</h3>
    """
    
    if feedback_to_system2:
        html += f"""
        <div style="
            background: #FFE0E0;
            padding: 12px;
            border-radius: 5px;
            margin-bottom: 15px;
            border-left: 3px solid #FF6B6B;
        ">
            <strong>Feedback to System 2:</strong>
            <div style="margin-top: 5px; font-style: italic;">{feedback_to_system2}</div>
        </div>
        """
    
    for constraint in blocking_constraints:
        constraint_type = constraint.get('type', 'unknown')
        severity = constraint.get('severity', 'hard')
        description = constraint.get('description', '')
        suggested_mitigation = constraint.get('suggested_mitigation', '')
        evidence_pointers = constraint.get('evidence_pointers', [])
        
        # Color coding based on severity
        severity_color = '#FF4444' if severity == 'hard' else '#FF8800'
        severity_badge = '🔴 HARD' if severity == 'hard' else '🟠 SOFT'
        
        html += f"""
        <div style="
            background: white;
            padding: 15px;
            border-radius: 5px;
            margin-bottom: 10px;
            border-left: 3px solid {severity_color};
        ">
            <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px;">
                <div style="font-weight: bold; font-size: 16px; color: {severity_color};">
                    {constraint_type.replace('_', ' ').title()}
                </div>
                <div style="
                    background: {severity_color};
                    color: white;
                    padding: 4px 8px;
                    border-radius: 3px;
                    font-size: 11px;
                    font-weight: bold;
                ">
                    {severity_badge}
                </div>
            </div>
            <div style="margin-top: 8px; line-height: 1.6;">
                {description}
            </div>
        """
        
        if suggested_mitigation:
            html += f"""
            <div style="margin-top: 10px; padding: 10px; background: #FFF9E6; border-radius: 3px;">
                <strong>Suggested Mitigation:</strong> {suggested_mitigation}
            </div>
            """
        
        if evidence_pointers:
            evidence_str = ', '.join([str(e) for e in evidence_pointers[:3]])  # Show first 3
            if len(evidence_pointers) > 3:
                evidence_str += f" ... and {len(evidence_pointers) - 3} more"
            html += f"""
            <div style="margin-top: 8px; font-size: 12px; color: #666;">
                <strong>Evidence:</strong> {evidence_str}
            </div>
            """
        
        html += "</div>"
    
    html += "</div>"
    display(HTML(html))

if system3_results and system3_results.get('status') == 'blocked':
    visualize_blocking_constraints(system3_results)